# nnUNet Results

In [1]:
import sys
import numpy as np
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed

repo_path = Path('~/ukbb-pulmonary-artery/DeepCMR')
assert repo_path.is_dir()
if not str(repo_path.resolve()) in sys.path: sys.path.append(str(rep o_path.resolve()))

from utils.strings import str2
import utils.strings as dstr

---
## Define paths

In [2]:
## ================================== EDIT THESE ==================================
## Location of our project folder
PROJ_ROOT = Path('{deepcmr_data_root}/nnUNET/')

## 1. Relative path to original niftis, for cross-checking
##    (created by notebooks/data/1_convert_dicoms_to_nifti.ipynb)
input_niftis1_root = PROJ_ROOT.joinpath('../cmr_lvot_20212_niftis/')
# input_niftis1_root = PROJ_ROOT.joinpath('../OrthancDicomStorage-PracticeNiftis/')

## 2. Relative path to nnUNet input niftis, for cross-checking
##    (for testing: created by notebooks/data/4_export_nifti_for_training_with_nnUNet.ipynb)
##    (for predicting: created by notebooks/data/5_export_nifti_for_predicting_with_nnUNet.ipynb)
input_niftis2_root = PROJ_ROOT.joinpath('exported_nifts_for_predicting/cmr_lvot_20212_niftis/')
# input_niftis2_root = PROJ_ROOT.joinpath('nnUNet_raw_data_base/nnUNet_raw_data/Task618_UKBBPulmonaryArtery/imagesTs')

## 3. Relative path to nnUNet predicitons
##    (created by nnUNet_test.sh or nnUNet_predict.sh or; for testing: nnUNET/results/test/taskID)
# predictions_root = 'nnUNet_predictions/Task142_UKBBPulmonaryArtery/cmr_lvot_20212/'
predictions_root = PROJ_ROOT.joinpath('nnUNet_predictions/Task618_UKBBPulmonaryArtery/cmr_lvot_20212')
# predictions_root = PROJ_ROOT.joinpath('results/test/618')

## 4. Output folder
##    (where to put compiled/final niftis, relative to project root)
# output_root = '../predicted_niftis_nnUNet/Task142_UKBBPulmonaryArtery/cmr_lvot_20212/'
output_path = PROJ_ROOT.joinpath('../predicted_niftis_nnUNet/Task618_UKBBPulmonaryArtery/cmr_lvot_20212/')

## Number of frames per patient
N_PER_PATIENT = 50
## ================================================================================

---
## Perform cross-checks
Gather files

In [3]:
# Get basenames of files in each folder
# Note: use rglob for exported and predicted files to recurse through batch folders
pnames_original = [ dstr.standardize_patient_name(str2(i).basename().rchop('.nii.gz'))
                        for i in input_niftis1_root.glob('*.nii.gz')
                        if not str(i).endswith('_gt.nii.gz') ]
pfiles_exported = [ str2(i).rchop('.nii.gz') for i in input_niftis2_root.rglob('*.nii.gz') ]
pfiles_predicted = [ str2(i).rchop('.nii.gz') for i in predictions_root.rglob('*.nii.gz') ]

# Check for non-unique files
if len(pnames_original) != len(np.unique(pnames_original)):
    raise Warning(" WARNING: There are non-unique patients in the ORIGINAL niftis folder.")
if len(pfiles_exported) != len(np.unique(pfiles_exported)):
    raise Warning(" WARNING: There are non-unique patients in the EXPORTED niftis folder.")
if len(pfiles_predicted) != len(np.unique(pfiles_predicted)):     raise Warning(" WARNING: There are non-unique patients in the PREDICTED niftis folder.")

# Print counts
pnames_original = np.unique(pnames_original)
pfiles_exported = np.unique(pfiles_exported)
pfiles_predicted = np.unique(pfiles_predicted)
print("Number of unique identifiers in ORIGINAL niftis folder: %.d" % (len(pnames_original)))
print("Number of unique identifiers in EXPORTED niftis folder: %.d" % (len(pfiles_exported)))
print("Number of unique identifiers in PREDICTED niftis folder: %.d" % (len(pfiles_predicted)))


Number of unique identifiers in ORIGINAL niftis folder: 44003
Number of unique identifiers in EXPORTED niftis folder: 2200150
Number of unique identifiers in PREDICTED niftis folder: 2200150


Compare original niftis to exported niftis & original niftis to predicted niftis.

In [4]:
def cross_check_files(pfiles_check, label):
    print("---")
    success = True

    # check num files
    if len(pfiles_check) == N_PER_PATIENT*len(pnames_original):
        print("** SUCCESS: Number of files in "+label+" niftis folder OK!")
    else:
        print("** WARNING: Incorrect number of files in "+label+" niftis folder (expected %.d, found %.d)" % (N_PER_PATIENT*len(pnames_original), len(pfiles_check)))

    # gather list of patient names & frame numbers
    id_list = {}
    for file in tqdm(pfiles_check, desc="Gathering list of patients from individual niftis", leave=False):

        if not dstr.validate_patient_filename_format(file):
            success = False
            raise Exception("** ERROR: " + file + " is not in the correct name format in " + label + " niftis folder")
            continue
        
        _, identifier, frame_num, index = dstr.deconstruct_patient_filename(file)

        if identifier not in id_list:
            id_list[identifier] = []
        if frame_num not in id_list[identifier]:
            id_list[identifier] += [frame_num]
    print("** SUCCESS: Name format of files in "+label+" niftis folder OK!")
        

    # check for completeness of each patient (check for frames 0 thru N)
    sb = True
    frame_nos = list(range(0,N_PER_PATIENT))
    for idn in tqdm(id_list, desc="Checking for completeness of each patient", leave=False):
        if set(id_list[idn]) != set(frame_nos):
            success = False
            raise Exception("** ERROR: " + idn + " is missing frames in " + label + " niftis folder")
    print("** SUCCESS: Completeness of patients in "+label+" niftis folder OK!")

    # check for same patient ids
    if set(id_list.keys()) != set(pnames_original):
        print("** WARNING: ORIGINAL patient IDs do not match IDs in " + label + " folder")
        success = False

        # check if subset of original
        if set(id_list.keys()).issubset(set(pnames_original)):
            print("** SUCCESS: Patients in "+label+" niftis folder is subset of ORIGINAL")
        else:
            raise Exception("** ERROR: Patients in "+label+" niftis folder is not a subset of ORIGINAL")
    else:
        print("** SUCCESS: Matching of ORIGINAL IDs to IDs of patients in "+label+" niftis folder OK!")

    print("\n" + label + " folder " + ("is OKAY" if success else "has ERRORS/WARNINGS") + "\n---\n")
    return success

In [5]:
export_check = cross_check_files(pfiles_exported, "EXPORTED")
predict_check = cross_check_files(pfiles_predicted, "PREDICTED")

---
** SUCCESS: Number of files in EXPORTED niftis folder OK!
** SUCCESS: Name format of files in EXPORTED niftis folder OK!
** SUCCESS: Completeness of patients in EXPORTED niftis folder OK!
** SUCCESS: Matching of ORIGINAL IDs to IDs of patients in EXPORTED niftis folder OK!

EXPORTED folder is OKAY
---

---
** SUCCESS: Number of files in PREDICTED niftis folder OK!
** SUCCESS: Name format of files in PREDICTED niftis folder OK!
** SUCCESS: Completeness of patients in PREDICTED niftis folder OK!
** SUCCESS: Matching of ORIGINAL IDs to IDs of patients in PREDICTED niftis folder OK!

PREDICTED folder is OKAY
---



---
## Compile niftis from individual predictions

Create an index and a function that will be paralellized.

In [6]:
output_path.mkdir(exist_ok=True, parents=True)

# gather names
names = []
input_filename_paths = []
input_filename_roots = []
output_filenames = []
if not predict_check:
    print("** WARNING: Warnings raised about files in prediction folder(s); compilation may fail. Recommend addressing/understanding them before proceeding.")
for file in tqdm(pfiles_predicted, desc="Indexing prediction files"):
    prefix, identifier, frame_num, index = dstr.deconstruct_patient_filename(file)
    if frame_num > 0: continue

    names += [ identifier ]
    input_filename_paths += [ Path(file).parent ]
    input_filename_roots += [ Path(file).name ]
    output_filenames += [ output_path.joinpath(prefix + '_' + identifier + '.nii.gz') ]

# define parallel function
def do_compile(i, input_filename_paths, input_filename_roots, output_filenames):

    filedir = input_filename_paths[i]
    filename = input_filename_roots[i]
    prefix, name, frame, index = dstr.deconstruct_patient_filename(filename)

    M_pred = []
    for t in range(50):
        filename_t = dstr.format_patient_filename(prefix, name, frame+t, index+t) + '.nii.gz'
        M_nifti = nib.load( filedir.joinpath(filedir, filename_t) )
        M_pred += [ M_nifti.get_fdata() ]
    M_pred_nifti = nib.Nifti1Image(np.stack(M_pred,-1), affine=M_nifti.affine)
    M_pred_nifti.to_filename( output_filenames[i] )

Indexing prediction files: 100%|██████████| 2200150/2200150 [00:14<00:00, 149382.77it/s]


Split the names into segments of 2000 each, and run each segment in parallel with 12 cores.

In [7]:
# run
num_cores = 12
pts_per_epoch = 2000
accumulator = 0
while accumulator < len(names):
    max_index = min(len(names), accumulator + pts_per_epoch)
    names_i = names[accumulator:max_index]
    inputpaths_i = input_filename_paths[accumulator:max_index]
    intputfilenames_i = input_filename_roots[accumulator:max_index]
    outputfilenames_i = output_filenames[accumulator:max_index]

    print("\nProcessing patients %.d through %.d, out of %.d total" % (accumulator+1, max_index, len(names)))
    Parallel(n_jobs=num_cores)(delayed(do_compile)(i, inputpaths_i, intputfilenames_i, outputfilenames_i)
        for i in tqdm(range(max_index-accumulator), unit=" patients", desc=("Compiling niftis using %.d cores" % num_cores)))

    accumulator += pts_per_epoch


Processing patients 1 through 2000, out of 44003 total
Compiling niftis using 12 cores: 100%|██████████| 2000/2000 [04:41<00:00,  7.10 patients/s]

Processing patients 2001 through 4000, out of 44003 total
Compiling niftis using 12 cores: 100%|██████████| 2000/2000 [04:40<00:00,  7.13 patients/s]

Processing patients 4001 through 6000, out of 44003 total
Compiling niftis using 12 cores: 100%|██████████| 2000/2000 [04:40<00:00,  7.13 patients/s]

Processing patients 6001 through 8000, out of 44003 total
Compiling niftis using 12 cores: 100%|██████████| 2000/2000 [04:41<00:00,  7.11 patients/s]

Processing patients 8001 through 10000, out of 44003 total
Compiling niftis using 12 cores: 100%|██████████| 2000/2000 [04:41<00:00,  7.10 patients/s]

Processing patients 10001 through 12000, out of 44003 total
Compiling niftis using 12 cores: 100%|██████████| 2000/2000 [04:40<00:00,  7.13 patients/s]

Processing patients 12001 through 14000, out of 44003 total
Compiling niftis using 12 cores: 